0. 제출 정보

- 이름: 예민호

- GitHub ID: yegamanse-cpu

- 작성일: 2026. 09. 30

- 최종 제출 URL: 

1. 질문과 필요한 데이터 선택

- 내가 확인하려는 질문 : 어떤 상품 카테고리의 completed 주문 기준 금액이 가장 큰가?

- 사용한 파일/컬럼 : products.csv ['product_name', 'category', 'price'], orders.csv ['order_id', 'order_status'], order_items ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']

- 결과 관찰 : 확인하고자 하는 데이터가 각 파일별로 흩어져 있음. 질문을 확인하기 위한 merge 필요

- 나의 해석과 판단 :  각 파일의 데이터를 merge하여 카테고리별 주문 기준 금액을 확인하고자 함

- 왜 이 컬럼과 범위를 선택했는지 작성하세요 : 제품의 이름과 카테고리, 금액은 기본 전제이고 주문번호와 상태. 단가와 물량이 필요하다 판단

- 업무·분석적 의미 : 각 파일이 필요한 데이터를 모두 포함하면 다행이겠지만 많은 경우 필요한 데이터가 흩어져 있기 떄문에 병합 및 분리 절차가 필수적

- 한계와 추가 확인 사항 : 현재 단계에는 칼럼명만 보고 사용여부 판단을 해보았으나 실제 데이터를 확인하고 사용 칼럼 재정리 추진

![01 캡처](images/01.png)

In [183]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)

Python 실행 파일: c:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
현재 작업 폴더: c:\dev\llm-data-analysis-course\notebooks\ch04
프로젝트 루트: C:\dev\llm-data-analysis-course
데이터 폴더: C:\dev\llm-data-analysis-course\data\raw
결과 저장 폴더: C:\dev\llm-data-analysis-course\reports


In [184]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')

데이터 불러오기 완료


In [185]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')

customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


In [186]:
#print([products,orders,order_items])
print(products.head())
print(products.info())

   product_id product_name category   price
0           1  전자기기 상품 001     전자기기  160000
1           2    도서 상품 002       도서   34000
2           3  전자기기 상품 003     전자기기  152000
3           4  생활용품 상품 004     생활용품   70000
4           5    식품 상품 005       식품  186000
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    100 non-null    int64
 1   product_name  100 non-null    str  
 2   category      100 non-null    str  
 3   price         100 non-null    int64
dtypes: int64(2), str(2)
memory usage: 3.3 KB
None


In [187]:
products["category"].value_counts(dropna=False)

category
스포츠     19
전자기기    17
생활용품    16
뷰티      16
도서      14
패션      11
식품       7
Name: count, dtype: int64

In [188]:
print(orders.head())
print(orders.info())

   order_id  customer_id  order_date payment_method order_status
0         1          123  2026-07-14           card    completed
1         2           77  2025-09-29      naver_pay    cancelled
2         3          138  2026-01-26  bank_transfer    cancelled
3         4           57  2026-04-08      kakao_pay    cancelled
4         5          125  2026-02-27           card    cancelled
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        300 non-null    int64
 1   customer_id     300 non-null    int64
 2   order_date      300 non-null    str  
 3   payment_method  300 non-null    str  
 4   order_status    300 non-null    str  
dtypes: int64(2), str(3)
memory usage: 11.8 KB
None


In [189]:
print(order_items.head())
print(order_items.info())

   order_item_id  order_id  product_id  quantity  unit_price
0              1         1         100         3      102000
1              2         1          87         5       25000
2              3         1           7         3      142000
3              4         1           9         3      193000
4              5         2          72         4      189000
<class 'pandas.DataFrame'>
RangeIndex: 764 entries, 0 to 763
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  764 non-null    int64
 1   order_id       764 non-null    int64
 2   product_id     764 non-null    int64
 3   quantity       764 non-null    int64
 4   unit_price     764 non-null    int64
dtypes: int64(5)
memory usage: 30.0 KB
None


2. 필터링·정렬·파생 컬럼

- 적용한 필터 조건: orders에서 complited된 데이터만 orders_f로 필터링

- 정렬 기준: product에서 catagory별 정렬은 시행했으나 orders의 status는 complited만 필터링 했으므로 별도의 정렬은 미시행 

- 만든 파생 컬럼: order_items [line_total]

- line_total 계산식: order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

- 필터와 파생 컬럼 : 필터 complited, 파생컬럼 [line_total]

- 결과 관찰 : order_items df에서 line_total coloum이 생성됨을 확인

- 나의 해석과 판단 : [quantity]와 [unit_price]값을 곱하여 [line_total]이 생성되었으나, 아직 주문완료 정보가 미포함 되었으므로 병합하여 확인필요

- 필터 기준이 달라지면 결과가 어떻게 달라질지 작성하세요 : 원하는 데이터를 얻지 못할 수 있으므로 정확한 필터기준을 생각하고 필터링 필요

- 업무·분석적 의미 : 데이터 분석을 위한 전처리 과정 일부를 직접 수행 중

- 한계와 추가 확인 사항 : 필터를 먼저하는게 맞는건지 병합을 시행한 후에 필터링을 하는게 맞는지 판단이 안되기 떄문에 다음 단계에서 병합 시도 후 확인 추진

![02 캡처](images/02.png)

In [190]:
orders_f = orders[orders["order_status"] == "completed"]
#print(orders_f.head())

print(orders)

print(orders_f)


     order_id  customer_id  order_date payment_method order_status
0           1          123  2026-07-14           card    completed
1           2           77  2025-09-29      naver_pay    cancelled
2           3          138  2026-01-26  bank_transfer    cancelled
3           4           57  2026-04-08      kakao_pay    cancelled
4           5          125  2026-02-27           card    cancelled
..        ...          ...         ...            ...          ...
295       296          116  2026-04-11           card    completed
296       297           22  2026-03-24  bank_transfer    completed
297       298           64  2026-03-03      naver_pay    cancelled
298       299          135  2025-10-15  bank_transfer    completed
299       300           96  2026-05-04      kakao_pay    cancelled

[300 rows x 5 columns]
     order_id  customer_id  order_date payment_method order_status
0           1          123  2026-07-14           card    completed
5           6           87  2026-05-28

In [191]:
products.sort_values("category", ascending=False)


,product_id,product_name,category,price
12,13,패션 상품 013,패션,164000
10,11,패션 상품 011,패션,115000
32,33,패션 상품 033,패션,124000
41,42,패션 상품 042,패션,28000
44,45,패션 상품 045,패션,62000
...,...,...,...,...
20,21,도서 상품 021,도서,37000
39,40,도서 상품 040,도서,174000
35,36,도서 상품 036,도서,126000
31,32,도서 상품 032,도서,172000


In [192]:
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]
print(order_items.head())

   order_item_id  order_id  product_id  quantity  unit_price  line_total
0              1         1         100         3      102000      306000
1              2         1          87         5       25000      125000
2              3         1           7         3      142000      426000
3              4         1           9         3      193000      579000
4              5         2          72         4      189000      756000


3. merge 검증

- 병합한 데이터: orders_f['order_id', 'customer_id', 'order_date', 'order_status']

- 사용한 key: order_id

- validate 결과: validate="many_to_one" 통과. orders["order_id"].duplicated().sum() == 0으로 order_id가 중복되지 않음을 확인했고, many_to_one 검증도 에러 없이 통과되어 order_id 1건에 매칭됨을 확인

- indicator 결과: _merge 값이 both 474건, left_only 290건

- 병합 전/후 행 수: 동일

- merge 검증 : indicator=True

결과 관찰
- 나의 해석과 판단 : merge=="both" 474건과 order_sales_f(=order_status=="completed"로 재필터링한 결과) 474건의 수가 정확히 일치함을 확인

- 이번 병합이 안전하다고 판단한 근거를 작성하세요. 행 수가 증가/감소했다면 이유를 설명하세요. 

   : 병합 전 orders["order_id"].duplicated()로 중복 여부 확인하고 indicator=True로 both/left_only건수를 대조했기 때문

   : 행수는 변동이 없었으나 order_f에서 complited항목을 필터링한 상태로 merge가 되어 실제론 결측된 상태로 병합이 됨. 이 결측된 부분을 재필터링 했기 때문에 결과적으론 행 수의 감소발생

업무·분석적 의미

- 잘못된 merge가 분석 결과에 어떤 영향을 줄 수 있는지 작성하세요. 

   : 사전에 중복 키를 확인하지 않고 병합했다면 order_id 중복으로 매칭이 일어나 line_total이 실제보다 부풀려 집계될 위험이 있음

- 한계와 추가 확인 사항 

   : 필터링을 먼저 하고 병합을 시도해도 자료자체는 문제없이 병합됨을 확인했으나 결측치가 발생하여 결국 다시 재필터링을 시행함(order_sales_f 생성 단계에서 재확인)

   ![03 캡처](images/03.png)

In [193]:
orders['order_id'].duplicated().sum()


np.int64(0)

In [194]:
order_sales = order_items.merge(
    orders_f[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print("전:",order_items.head(10))

print("후:",order_sales.head(10))

print(order_sales["_merge"].value_counts())

전:    order_item_id  order_id  product_id  quantity  unit_price  line_total
0              1         1         100         3      102000      306000
1              2         1          87         5       25000      125000
2              3         1           7         3      142000      426000
3              4         1           9         3      193000      579000
4              5         2          72         4      189000      756000
5              6         3          33         5      124000      620000
6              7         3          20         3       80000      240000
7              8         3          11         5      115000      575000
8              9         4          45         3       62000      186000
9             10         4          84         4       78000      312000
후:    order_item_id  order_id  product_id  quantity  unit_price  line_total  \
0              1         1         100         3      102000      306000   
1              2         1          87 

- 위의 코드를 실행하였을 때 left_only가 발생하는 이유는 앞서 필터링을 수행한 자료(orders_f)를 병합하였기 때문.

- 필터링을 재수행하면 분석에 문제없을 것으로 판단됨

In [195]:
order_sales = order_sales.drop(columns="_merge")

In [196]:
print(order_sales)

     order_item_id  order_id  product_id  quantity  unit_price  line_total  \
0                1         1         100         3      102000      306000   
1                2         1          87         5       25000      125000   
2                3         1           7         3      142000      426000   
3                4         1           9         3      193000      579000   
4                5         2          72         4      189000      756000   
..             ...       ...         ...       ...         ...         ...   
759            760       297          42         4       28000      112000   
760            761       298          40         4      174000      696000   
761            762       299           8         2      189000      378000   
762            763       299          12         4      175000      700000   
763            764       300          59         5       32000      160000   

     customer_id  order_date order_status  
0          123.0  2

In [197]:
order_sales_f = order_sales[order_sales["order_status"] == "completed"]

print(order_sales_f)

     order_item_id  order_id  product_id  quantity  unit_price  line_total  \
0                1         1         100         3      102000      306000   
1                2         1          87         5       25000      125000   
2                3         1           7         3      142000      426000   
3                4         1           9         3      193000      579000   
12              13         6          83         3       24000       72000   
..             ...       ...         ...       ...         ...         ...   
757            758       296          40         1      174000      174000   
758            759       297          20         5       80000      400000   
759            760       297          42         4       28000      112000   
761            762       299           8         2      189000      378000   
762            763       299          12         4      175000      700000   

     customer_id  order_date order_status  
0          123.0  2

In [198]:
order_sales_f["order_status"].value_counts()

order_status
completed    474
Name: count, dtype: int64

- 필터링 재수행 후 474열이 있으며 결측치가 없는것을 확인했다.

In [199]:
order_sales_f["order_date"] = pd.to_datetime(order_sales["order_date"], errors="coerce",)

print("날짜 변환 실패:", order_sales_f["order_date"].isna().sum(), )


날짜 변환 실패: 0


In [200]:
order_sales_f["order_month"] = order_sales_f["order_date"].dt.to_period("M").astype(str) 

print(order_sales_f.info())

<class 'pandas.DataFrame'>
Index: 474 entries, 0 to 762
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_item_id  474 non-null    int64         
 1   order_id       474 non-null    int64         
 2   product_id     474 non-null    int64         
 3   quantity       474 non-null    int64         
 4   unit_price     474 non-null    int64         
 5   line_total     474 non-null    int64         
 6   customer_id    474 non-null    float64       
 7   order_date     474 non-null    datetime64[us]
 8   order_status   474 non-null    str           
 9   order_month    474 non-null    str           
dtypes: datetime64[us](1), float64(1), int64(6), str(2)
memory usage: 40.7 KB
None


In [201]:
print(order_sales_f.head(10))

    order_item_id  order_id  product_id  quantity  unit_price  line_total  \
0               1         1         100         3      102000      306000   
1               2         1          87         5       25000      125000   
2               3         1           7         3      142000      426000   
3               4         1           9         3      193000      579000   
12             13         6          83         3       24000       72000   
13             14         6          17         5       84000      420000   
14             15         6          12         4      175000      700000   
15             16         6          66         3       39000      117000   
19             20         9          29         2       78000      156000   
20             21         9          20         1       80000       80000   

    customer_id order_date order_status order_month  
0         123.0 2026-07-14    completed     2026-07  
1         123.0 2026-07-14    completed     

In [202]:
print("전:", len(order_sales_f))

order_sales_f = order_sales_f.merge(products, on="product_id", how="left", validate="many_to_one", indicator=True, )

print("후:", len(order_sales_f))

print(order_sales_f["_merge"].value_counts())
print(order_sales_f)


전: 474
후: 474
_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64
     order_item_id  order_id  product_id  quantity  unit_price  line_total  \
0                1         1         100         3      102000      306000   
1                2         1          87         5       25000      125000   
2                3         1           7         3      142000      426000   
3                4         1           9         3      193000      579000   
4               13         6          83         3       24000       72000   
..             ...       ...         ...       ...         ...         ...   
469            758       296          40         1      174000      174000   
470            759       297          20         5       80000      400000   
471            760       297          42         4       28000      112000   
472            762       299           8         2      189000      378000   
473            763       299          12 

In [211]:
category_sales = (order_sales_f.groupby("category", as_index=False)

    .agg(total_quantity=("quantity", "sum"), total_sales=("line_total", "sum"),)

    .sort_values("total_sales", ascending=False))

In [204]:
category_sales["sales_ratio"] = (

    category_sales["total_sales"]

    / category_sales["total_sales"].sum()

    * 100

).round(2)

In [205]:

product_sales = (

    order_sales_f

    .groupby(

        ["product_id", "product_name", "category"],

        as_index=False,

    )

    .agg(

        total_quantity=("quantity", "sum"),

        total_sales=("line_total", "sum"),

    )

    .sort_values("total_sales", ascending=False)

)

In [206]:
monthly_summary = (

    order_sales_f

    .groupby("order_month", as_index=False)

    .agg(

        total_sales=("line_total", "sum"),

        order_count=("order_id", "nunique"),

    )

    .sort_values("order_month")

)

In [207]:
monthly_summary["average_order_value"] = (

    monthly_summary["total_sales"]

    / monthly_summary["order_count"]

).round(0)

In [209]:
print(order_sales_f["order_date"].min())

print(order_sales_f["order_date"].max())

2025-09-15 00:00:00
2026-09-11 00:00:00


In [212]:
customer_sales = (

    order_sales_f

    .groupby("customer_id", as_index=False)

    .agg(

        order_count=("order_id", "nunique"),

        total_sales=("line_total", "sum"),

    )

    .sort_values("total_sales", ascending=False)

)

In [213]:

customer_sales = customer_sales.merge(

    customers[["customer_id", "city"]],

    on="customer_id",

    how="left",

    validate="one_to_one",

)

In [214]:
customer_sales["customer_label"] = (

    "Customer "

    + customer_sales["customer_id"].astype(str)

)

In [216]:
base_total = order_sales_f["line_total"].sum()

category_total = category_sales["total_sales"].sum()

product_total = product_sales["total_sales"].sum()

monthly_total = monthly_summary["total_sales"].sum()

customer_total = customer_sales["total_sales"].sum()

print("base:", base_total)

print("category:", category_total)

print("product:", product_total)

print("month:", monthly_total)

print("customer:", customer_total)

base: 148990000
category: 148990000
product: 148990000
month: 148990000
customer: 148990000


In [217]:
category_sales.to_csv(

    REPORT_DIR / "ch04_category_sales.csv",

    index=False,

    encoding="utf-8-sig",

)

product_sales.to_csv(

    REPORT_DIR / "ch04_product_sales.csv",

    index=False,

    encoding="utf-8-sig",

)

monthly_summary.to_csv(

    REPORT_DIR / "ch04_monthly_sales.csv",

    index=False,

    encoding="utf-8-sig",

)

customer_sales.to_csv(

    REPORT_DIR / "ch04_customer_sales.csv",

    index=False,

    encoding="utf-8-sig",

)

In [218]:

for path in sorted(REPORT_DIR.glob("ch04_*.csv")):

    print(path.name, path.exists(), path.stat().st_size)

ch04_category_sales.csv True 202
ch04_customer_sales.csv True 3787
ch04_monthly_sales.csv True 443
ch04_product_sales.csv True 4409


In [219]:

saved_category_sales = pd.read_csv(

    REPORT_DIR / "ch04_category_sales.csv"

)

print(saved_category_sales.shape)

print(saved_category_sales.columns.tolist())

saved_category_sales.head()

(7, 3)
['category', 'total_quantity', 'total_sales']


,category,total_quantity,total_sales
0,스포츠,295,31743000
1,전자기기,259,26400000
2,생활용품,272,23915000
3,뷰티,223,23383000
4,식품,133,16573000


4. completed 주문 범위와 집계
- 분석 범위 정의:  order_sales_f를 기준으로 집계

- 카테고리별 결과: category_sales — 스포츠 31,743,000원(295개), 전자기기 26,400,000원(259개), 생활용품 23,915,000원(272개), 뷰티 23,383,000원(223개), 식품 16,573,000원(133개), 도서 16,389,000원, 패션 10,587,000원

- 상품별 결과: product_id 41(스포츠) 5,705,000원, product_id 12(식품) 4,375,000원, product_id 9(스포츠) 3,860,000원, product_id 72(뷰티) 3,780,000원, product_id 71(전자기기) 3,703,000원

- 월별 결과: order_date 범위 2025-09-15 ~ 2026-09-11(13개월). 월별 매출 2025-12 21,085,000원 최고. 2025-09 2,650,000 최저. 

- 고객별 결과: customer_id 117 4,100,000원 최고, 102가 3,996,000원, 83이 3,880,000원

- 핵심 집계 결과 : base_total(order_sales_f["line_total"].sum()) = 148,990,000원. category_total / product_total / monthly_total / customer_total groupby 결과의 합 동일(148,990,000원)

결과 관찰

- 수치에서 직접 확인한 상위/하위 또는 변화 패턴을 작성하세요.
   : 카테고리 1위(스포츠)와 7위(패션)의 매출 차이가 31,743,000원 vs 10,587,000원으로 약 3배

나의 해석과 판단

- 어떤 결과가 가장 중요하다고 판단했는지와 이유를 작성하세요.
   : "어떤 카테고리의 completed 주문 금액이 가장 큰가"에 대한 답은 스포츠가 1위. 다만 상품별 결과를 같이 보면 스포츠 매출이 일부 상품(특히 product_id 41, 9)에 의존하고 있어, 카테고리 순위만 보고 "스포츠 카테고리 상품들이 전반적으로 잘 팔린다"고 단정하기는 어렵다고 판단됨

업무·분석적 의미

- 다음 의사결정/분석으로 어떻게 연결할지 작성하세요.
   : 스포츠·전자기기처럼 매출 상위 카테고리는 재고·마케팅 우선순위로 검토하되, 카테고리 내에서도 특정 상품에 매출이 쏠려 있는지 상품별 결과로 다시 확인해야 함. 

한계와 추가 확인 사항
- total_sales가 회계상 순매출과 같다고 단정할 수 없는 이유를 포함하세요.
   : payment_method별로 취소/환불 처리 방식이 다를 수 있는데 이 데이터만으로는 확인이 안 되므로, 실제 업무에 쓰려면 재무팀 데이터와 대조가 추가로 필요함

   ![04 캡처](images/04.png)

5. 총합 일치 검증

- 원본 completed line_total 합계: base_total = order_sales_f["line_total"].sum() = 148,990,000원

- 카테고리 합계:  category_total = category_sales["total_sales"].sum() = 148,990,000원

- 월별 합계:  monthly_total = monthly_summary["total_sales"].sum() = 148,990,000원

- 고객별 합계: customer_total = customer_sales["total_sales"].sum() = 148,990,000원
 
- 차이 여부: base / category / product / month / customer 5개 합계가 전부148,990,000원으로 동일

- 총합 검증: 이상없음 

나의 해석과 판단

- 총합 검증이 필요한 이유와 불일치 시 확인할 순서를 작성하세요.

   : groupby로 쪼갠 뒤 다시 더한 값이 쪼개기 합계와 같아야, group 기준 컬럼(category/order_month/customer_id)에 결측치가 없고 groupby 과정에서 누락되거나 중복 집계되지 않았다는 것을 보장 가능
  
   : (1) groupby 기준 컬럼에 NaN이 있는지 확인 → (2) merge 단계에서 키 중복으로 행이 불어나지는 않았는지 확인 → (3) 각 집계 전에 order_status=="completed" 필터가 빠짐없이 동일하게 적용되었는지 확인

    ![05 캡처](images/05.png)

6. LLM pandas 코드 검증
- LLM Prompt 요약
   : orders와 order_items를 사용해 completed 주문 기준 월별 금액을 계산하려고 합니다. 반드시 다음 조건을 포함해 주세요.

1. quantity × unit_price로 line_total 생성
2. orders.order_id 고유성 확인
3. many-to-one merge와 indicator 사용
4. 병합 전후 행 수와 미매칭 확인
5. 날짜 변환 실패 건수 확인
6. order_status == completed만 필터링
7. 월별 total_sales와 고유 order_count 계산
8. 원본 completed total과 월별 합계를 비교
9. 결과를 회계상 순매출이라고 단정하지 않기

- 제안 코드 요약
   : order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

   : order_sales = order_items.merge(
    orders[["order_id", "customer_id", "order_date", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,)  등등

- 실제 컬럼/범위와 맞지 않은 부분: 없음

- 수정한 내용: 없음

- 최종 판단: 사용 / 수정 후 사용 / 보류

- LLM 코드 검증 : order_id 중복 0건, completed 행 수 474건, 날짜 변환 실패 0건 — 동일, 월별 집계 (2025-09 ~ 2026-02 구간) 동일

나의 해석과 판단

- LLM 코드가 실행되었다는 것만으로 분석이 맞다고 볼 수 없는 이유를 작성하세요.
   : 코드는 얼마든지 실행될 수 있으나 실제 아웃풋이 맞는 결과를 가져오는지 일일이 따져봐야 한다. 금번 LLM의 코딩은 문제가 없었지만 실제 없는 칼럼이나 범위를 지정한다면 전혀 다른 결과를 가져올 수 있다.

    ![06 캡처](images/06.png)

7. Chapter 04 최종 인사이트

- 가장 의미 있다고 생각한 결과 2가지
   : 카테고리별 completed 매출 1위는 스포츠(31,743,000원)지만, 그 매출의 상당 부분이 소수 상품에 몰려 있다는 점. 카테고리 순위만 보고 "그 카테고리 상품이 골고루 잘 팔린다"고 볼 수는 없다는 것을 확인
   : 월별 매출 변동폭이 카테고리 간 변동폭보다 훨씬 크다는 점으로 보아 매출을 좌우하는 요인이 "어떤 것을 파는가"보다 "언제 파는가"일 가능성이 더 큼

- 그 결과를 뒷받침하는 수치/표
   : category_sales : 스포츠 31,743,000원(295개) > 전자기기 26,400,000원 > 생활용품 23,915,000원 > 뷰티 23,383,000원 > 식품 16,573,000원 > 도서 16,389,000원 > 패션 10,587,000원
   : product_sales : 1위 product_id 41(스포츠) 5,705,000원 (총매출의 약 18%)
   : monthly_summary : 2025-12 21,085,000원(최고) vs 2025-09 2,650,000원(최저)

- 추가로 확인하고 싶은 질문 : 스포츠 카테고리에서 product_id 41, 9 두 상품을 빼도 카테고리 순위 1위가 유지되는지

- 현재 결과의 한계 : 데이터 기간이 1년 주기를 온전히 두 번 이상 보기엔 짧아서, 지금 단계에서 "계절성"이라고 단정하기는 이름


최종 제출 체크

- 핵심 셀 Output이 남아 있습니다.

- merge와 총합 검증 Evidence가 있습니다.

- 결과 관찰과 해석이 구분되어 있습니다.

- LLM 코드를 검증했습니다.

- 개인정보/Secret이 없습니다.

- chapter04/chapter04.ipynb가 GitHub에서 정상 표시됩니다.

- 최종 Notebook 파일 URL을 제출합니다.